# Train detector MŨ BẢO HIỂM (With Helmet / Without Helmet) trên Kaggle (~3.5 tiếng)

**Vì sao:** `helmet_best.pt` cũ (YOLOv8n train ở 224 px, 25 epoch) nhầm **tóc đen thành mũ**. Trên val ảnh camera thật nó chỉ đạt mAP50 0.357 (lớp "With Helmet" 0.18).

Dữ liệu `helmet_mix.zip` (tạo bằng `python scripts/training/build_helmet_dataset.py`, zip ảnh thu về cạnh dài ≤ 1280):
- **EdgeVision** (CC BY 4.0, Gajjar et al. 2026, doi:10.17632/j82bnw7gsr.1) — Helmet/NoHelmet đổi tên lớp, bỏ BikeWithRider.
- **Khung hình camera cổng trường** đã duyệt tay: 268 có mũ, 150 không mũ, 145 vùng nền (áp phích ATGT vẽ người đội mũ, đồ vật), vùng không chắc đã che xám.
- **Đầu trần thật** từ video camera trước 04/10.

**`best.pt` chọn theo mAP trên VAL ẢNH THẬT** (video không có trong train). Tên lớp giữ đúng `With Helmet`, `Without Helmet` (app kiểm tra tên lớp).

## Cách chạy
1. **Settings → Accelerator → GPU T4 x2** (hoặc P100). **Settings → Internet → On**.
2. **+ Add Input → Upload** `datasets/helmet_mix.zip` — **bắt buộc**.
3. *(Tùy chọn)* **+ Add Input → Upload** `models/helmet_best.pt` để notebook in số so sánh với model cũ.
4. **Save Version → Save & Run All (Commit)**. Không chạy interactive.
5. Xong: tab **Output** → tải `helmet_new_best.pt` + `helmet_new_metrics.json`.

In [ ]:
# ===== CẤU HÌNH =====
TRAIN_HOURS = 3.0       # còn ~30 phút cho cài đặt + val + xuất file
MODEL = "yolo11n.pt"    # app chạy 3 model song song trên GPU 4 GB -> giữ bản n; "yolo11s.pt" chính xác hơn nhưng chậm ~2.5x
IMGSZ = 640             # = DETECT_WIDTH của app
BATCH = 32              # CUDA out of memory -> 16
RESUME = True

In [ ]:
import os, sys
os.environ["WANDB_MODE"] = "disabled"
os.environ["WANDB_DISABLED"] = "true"
os.environ["ULTRALYTICS_TELEMETRY"] = "False"
# Kaggle có cudf-polars-cu12 cần polars >=1.30,<1.36 — không dùng `-U polars`.
!{sys.executable} -m pip install -q --upgrade --upgrade-strategy only-if-needed ultralytics "polars>=1.30,<1.36"

import torch, ultralytics
from ultralytics import settings
for k in ("raytune", "wandb", "comet", "clearml", "mlflow", "neptune", "dvc"):
    try:
        settings.update({k: False})
    except Exception:
        pass
assert torch.cuda.is_available(), "KHÔNG có GPU: Settings -> Accelerator -> GPU T4 x2, rồi Restart Session"
print("Ultralytics", ultralytics.__version__, "| Torch", torch.__version__, "| GPU", torch.cuda.get_device_name(0))

## 1. Dữ liệu

In [ ]:
import json, shutil, zipfile
from pathlib import Path

INPUT, WORK = Path("/kaggle/input"), Path("/kaggle/working")

def find_data():
    for root in (INPUT, WORK / "data"):
        if root.exists():
            for p in root.rglob("helmet_mix_meta.json"):
                return p.parent
    return None

data_dir = find_data()
if data_dir is None:  # upload dạng zip mà Kaggle không tự giải nén
    zips = [p for p in INPUT.rglob("*.zip") if "helmet" in p.name.lower()] if INPUT.exists() else []
    assert zips, "Thiếu Input helmet_mix.zip"
    zipfile.ZipFile(zips[0]).extractall(WORK / "data")
    data_dir = find_data()
assert data_dir, "Không thấy helmet_mix_meta.json"

# Input của Kaggle chỉ đọc; yaml viết ra /kaggle/working, trỏ path vào data_dir
DATA = {}
for val in ("val_real", "val_public"):
    p = WORK / f"data_{val}.yaml"
    p.write_text(f"path: {data_dir}\ntrain: images/train\nval: images/{val}\nnc: 2\n"
                 "names: ['With Helmet', 'Without Helmet']\n")
    DATA[val] = str(p)
print("Dữ liệu:", data_dir)
print(json.dumps(json.loads((data_dir / "helmet_mix_meta.json").read_text()), indent=1))

## 2. Mốc so sánh: model cũ (nếu có upload)

In [ ]:
from ultralytics import YOLO

def score(model_path, data):
    r = YOLO(model_path).val(data=data, imgsz=IMGSZ, batch=16, device=0, plots=False, verbose=False)
    names = ["With Helmet", "Without Helmet"]
    return {"P": round(float(r.box.mp), 4), "R": round(float(r.box.mr), 4),
            "mAP50": round(float(r.box.map50), 4), "mAP50-95": round(float(r.box.map), 4),
            "mAP50_per_class": {names[i]: round(float(v), 4) for i, v in zip(r.box.ap_class_index, r.box.ap50)}}

old = sorted(p for p in INPUT.rglob("*.pt") if "helmet" in p.name.lower()) if INPUT.exists() else []
baseline = {v: score(str(old[0]), d) for v, d in DATA.items()} if old else None
print("Model cũ:", old[0] if old else "không upload", "\n", json.dumps(baseline, indent=1))

## 3. Train (tự dừng sau `TRAIN_HOURS`; `best.pt` theo val ảnh thật)

In [ ]:
RUN = WORK / "runs" / "helmet_new"
last = RUN / "weights" / "last.pt"
if RESUME and last.exists():
    try:
        YOLO(str(last)).train(resume=True)
    except Exception as e:
        print("Không resume được:", e)
else:
    YOLO(MODEL).train(
        data=DATA["val_real"], epochs=1000, time=TRAIN_HOURS, patience=50,
        imgsz=IMGSZ, batch=BATCH, device=0, workers=4, cache="ram",
        project=str(RUN.parent), name=RUN.name, exist_ok=True, seed=42, plots=True,
        close_mosaic=10,
        fliplr=0.5, degrees=5, hsv_v=0.5,   # camera cổng: ngược sáng/bóng râm
    )

## 4. So sánh + xuất file

In [ ]:
best = RUN / "weights" / "best.pt"
assert best.exists(), f"Không có {best} — xem log ô train"
m = YOLO(str(best))
assert set(m.names.values()) == {"With Helmet", "Without Helmet"}, m.names
result = {v: score(str(best), d) for v, d in DATA.items()}
for name, res in (("CŨ", baseline), ("MỚI", result)):
    if res:
        for v in DATA:
            r = res[v]
            print(f"{name:4s} {v:11s} P={r['P']:.3f} R={r['R']:.3f} mAP50={r['mAP50']:.3f} "
                  f"mAP50-95={r['mAP50-95']:.3f} {r['mAP50_per_class']}")
shutil.copy2(best, WORK / "helmet_new_best.pt")
(WORK / "helmet_new_metrics.json").write_text(json.dumps(
    {"model": MODEL, "imgsz": IMGSZ, "baseline": baseline, "new": result}, indent=1))
shutil.make_archive(str(WORK / "helmet_new_run"), "zip", RUN)
shutil.rmtree(WORK / "data", ignore_errors=True)
for f in sorted(WORK.glob("helmet_new_*")):
    print(f.name, f"{f.stat().st_size/1e6:.1f} MB")

## 5. Đưa về máy
1. Tải `helmet_new_best.pt` → lưu `models/helmet_new_best.pt` (**đừng ghi đè** `helmet_best.pt`, test khoá hash).
2. Báo Claude: đo trên video thật (tỉ lệ đầu trần bị gọi "có mũ") rồi mới bật bằng env `HELMET_MODEL_PATH`.